In [8]:

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from matplotlib.ticker import MultipleLocator
import re
from tqdm import tqdm
import scipy

# Adquiriendo los Datos

In [9]:
#### Abriendo lista de top 40 en SPLAC
with open('SPLAC_Companias.txt', 'r', encoding='utf-8') as file:
    tickers = file.readlines()

tickers = [elem.rstrip('\n') for elem in tickers]
tickers = sorted(tickers)
    # Necesario pq yf automaticamente ordena alfabeticamente``
print(tickers)

['ABEV', 'AC.MX', 'AMXB.MX', 'AXIA3.SA', 'B3SA3.SA', 'BAP', 'BBAS3.SA', 'BBD', 'BIMBOA.MX', 'BSAC', 'CEMEXCPO.MX', 'CENCOSUD.SN', 'CHILE.SN', 'CIB', 'EC', 'FALABELLA.SN', 'FEMSAUBD.MX', 'FUNO11.MX', 'GCARSOA1.MX', 'GFNORTEO.MX', 'GGB', 'GMEXICOB.MX', 'ISA.CL', 'ITSA4.SA', 'ITUB', 'LTM', 'NU', 'PAC', 'PBR', 'PBR-A', 'RDOR3.SA', 'RENT3.SA', 'SBSP3.SA', 'SCCO', 'SQM', 'USD', 'VALE', 'VIV', 'WALMEX.MX', 'WEGE3.SA']


In [10]:
#### Descargar los datos inciales
# start_date = "2003-01-01"
# end_date = "2026-05-29"
# df_crudo = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)
#     # OJO: al descargar, yf automaticamente los ordena alfabeticamente
#         # Entonces se va a tener que orden "tickers" con nombres SPLAc originales
# df_crudo.to_csv('datos_crudos_splac.csv', index=True, encoding='utf-8')

In [11]:
#### Lectura y Limpieza

### Cargando
df_crudos = pd.read_csv('datos_crudos_splac_v1.csv')
print(df_crudos.shape)



### Limpieza
## Arreglando nombres de TODAS las columnas
nombres = df_crudos.columns.to_list()
print(nombres)  #Columnas pre-cambio

for i in range(1,len(nombres)):
    nombres[i] = tickers[(i-1)%len(tickers)] + '.' + nombres[i]

print(nombres)  #Columnas post-cambios

nombres[0] = 'Date' #Primera columna son fechas
df_crudos.columns = nombres #Aplicando los cambios



### Eliminando primera fila, redundancia
    # Primera fila es una repeticion de nombre de acciones
df_crudos = df_crudos.drop(index=0).reset_index(drop=True)
    #"reset_index": reinicia los indices



### Cambiando el index del conjunto
df_crudos = df_crudos.set_index('Date',drop=True)
    #"drop=True": elimina la columna de "Date" al cambiarlo a index


### Verificacion
# display(df_crudos.head())   # Visualizar conjunto
df_crudos.info()            # INformacion general
df_crudos.iloc[0]           # Primera fila
df_crudos.index[0]          # Nombre del primer index del cojunto
# print(df_crudos.shape)



(6105, 241)
['Price', 'Adj Close', 'Adj Close.1', 'Adj Close.2', 'Adj Close.3', 'Adj Close.4', 'Adj Close.5', 'Adj Close.6', 'Adj Close.7', 'Adj Close.8', 'Adj Close.9', 'Adj Close.10', 'Adj Close.11', 'Adj Close.12', 'Adj Close.13', 'Adj Close.14', 'Adj Close.15', 'Adj Close.16', 'Adj Close.17', 'Adj Close.18', 'Adj Close.19', 'Adj Close.20', 'Adj Close.21', 'Adj Close.22', 'Adj Close.23', 'Adj Close.24', 'Adj Close.25', 'Adj Close.26', 'Adj Close.27', 'Adj Close.28', 'Adj Close.29', 'Adj Close.30', 'Adj Close.31', 'Adj Close.32', 'Adj Close.33', 'Adj Close.34', 'Adj Close.35', 'Adj Close.36', 'Adj Close.37', 'Adj Close.38', 'Adj Close.39', 'Close', 'Close.1', 'Close.2', 'Close.3', 'Close.4', 'Close.5', 'Close.6', 'Close.7', 'Close.8', 'Close.9', 'Close.10', 'Close.11', 'Close.12', 'Close.13', 'Close.14', 'Close.15', 'Close.16', 'Close.17', 'Close.18', 'Close.19', 'Close.20', 'Close.21', 'Close.22', 'Close.23', 'Close.24', 'Close.25', 'Close.26', 'Close.27', 'Close.28', 'Close.29', 'C

C:\Users\JP\AppData\Local\Temp\ipykernel_464\3110181244.py:4: DtypeWarning: Columns (0: Adj Close, 1: Adj Close.1, 2: Adj Close.2, 3: Adj Close.3, 4: Adj Close.4, 5: Adj Close.5, 6: Adj Close.6, 7: Adj Close.7, 8: Adj Close.8, 9: Adj Close.9, 10: Adj Close.10, 11: Adj Close.11, 12: Adj Close.12, 13: Adj Close.13, 14: Adj Close.14, 15: Adj Close.15, 16: Adj Close.16, 17: Adj Close.17, 18: Adj Close.18, 19: Adj Close.19, 20: Adj Close.20, 21: Adj Close.21, 22: Adj Close.22, 23: Adj Close.23, 24: Adj Close.24, 25: Adj Close.25, 26: Adj Close.26, 27: Adj Close.27, 28: Adj Close.28, 29: Adj Close.29, 30: Adj Close.30, 31: Adj Close.31, 32: Adj Close.32, 33: Adj Close.33, 34: Adj Close.34, 35: Adj Close.35, 36: Adj Close.36, 37: Adj Close.37, 38: Adj Close.38, 39: Adj Close.39, 40: Close, 41: Close.1, 42: Close.2, 43: Close.3, 44: Close.4, 45: Close.5, 46: Close.6, 47: Close.7, 48: Close.8, 49: Close.9, 50: Close.10, 51: Close.11, 52: Close.12, 53: Close.13, 54: Close.14, 55: Close.15, 56: C

'2003-01-01'

In [12]:
#### Seleccionando columnas adecuadas
bool_lista = df_crudos.columns.str.contains(r'Adj Close')
    # Solo nos vamos a enfocar en 'Adj Close'
precios_nombres = []
for col_nom, bol in zip(df_crudos.columns.to_list(), bool_lista):
    if bol: precios_nombres.append(col_nom)

print(precios_nombres)

['ABEV.Adj Close', 'AC.MX.Adj Close.1', 'AMXB.MX.Adj Close.2', 'AXIA3.SA.Adj Close.3', 'B3SA3.SA.Adj Close.4', 'BAP.Adj Close.5', 'BBAS3.SA.Adj Close.6', 'BBD.Adj Close.7', 'BIMBOA.MX.Adj Close.8', 'BSAC.Adj Close.9', 'CEMEXCPO.MX.Adj Close.10', 'CENCOSUD.SN.Adj Close.11', 'CHILE.SN.Adj Close.12', 'CIB.Adj Close.13', 'EC.Adj Close.14', 'FALABELLA.SN.Adj Close.15', 'FEMSAUBD.MX.Adj Close.16', 'FUNO11.MX.Adj Close.17', 'GCARSOA1.MX.Adj Close.18', 'GFNORTEO.MX.Adj Close.19', 'GGB.Adj Close.20', 'GMEXICOB.MX.Adj Close.21', 'ISA.CL.Adj Close.22', 'ITSA4.SA.Adj Close.23', 'ITUB.Adj Close.24', 'LTM.Adj Close.25', 'NU.Adj Close.26', 'PAC.Adj Close.27', 'PBR.Adj Close.28', 'PBR-A.Adj Close.29', 'RDOR3.SA.Adj Close.30', 'RENT3.SA.Adj Close.31', 'SBSP3.SA.Adj Close.32', 'SCCO.Adj Close.33', 'SQM.Adj Close.34', 'USD.Adj Close.35', 'VALE.Adj Close.36', 'VIV.Adj Close.37', 'WALMEX.MX.Adj Close.38', 'WEGE3.SA.Adj Close.39']


In [13]:
#### Copiando datos para mantener datos originales
df_precios=df_crudos[precios_nombres].copy()

print(len(df_precios))
print(df_precios.isna().sum())
    # Viendo cuantos valores faltantes existen

6104
ABEV.Adj Close                216
AC.MX.Adj Close.1             189
AMXB.MX.Adj Close.2           189
AXIA3.SA.Adj Close.3          259
B3SA3.SA.Adj Close.4         1494
BAP.Adj Close.5               216
BBAS3.SA.Adj Close.6          259
BBD.Adj Close.7               216
BIMBOA.MX.Adj Close.8         189
BSAC.Adj Close.9              216
CEMEXCPO.MX.Adj Close.10      189
CENCOSUD.SN.Adj Close.11      214
CHILE.SN.Adj Close.12        1508
CIB.Adj Close.13              216
EC.Adj Close.14              1654
FALABELLA.SN.Adj Close.15     214
FEMSAUBD.MX.Adj Close.16      189
FUNO11.MX.Adj Close.17       2292
GCARSOA1.MX.Adj Close.18      189
GFNORTEO.MX.Adj Close.19      189
GGB.Adj Close.20              216
GMEXICOB.MX.Adj Close.21      189
ISA.CL.Adj Close.22            31
ITSA4.SA.Adj Close.23         259
ITUB.Adj Close.24             216
LTM.Adj Close.25             5642
NU.Adj Close.26              4984
PAC.Adj Close.27             1009
PBR.Adj Close.28              216
PBR-A.Adj

In [14]:
#### Limpiando datos
### Copiando datos
# df_precios=df_crudos[precios_nombres].copy()

### Cambiando nombre de columnas
    #Ej: "AMXB.MX.Adj Close.2" -> "AMXB"
    #Ojo: tickers = "AMXB.MX" -> "AMXB"
        # tickers ya esta ordenado
tickers = [elem.split('.')[0] for elem in tickers]
df_precios.columns = tickers

### Eliminando columnas que no tenga suficiente datos
max_dias = len(df_precios)
THRESH = 0.2    #Cantidad de dias/datos originales dispuesto a borrar, 0.2 => 20%
for col in df_precios.columns:
    if df_precios[col].isna().sum() > THRESH * max_dias:
        df_precios = df_precios.drop(col, axis=1)


### Hacer forward fill y eliminar Nans
df_precios = df_precios.ffill().dropna()

### Transformat tipos de datos de str a float
df_precios = df_precios.astype(float)

print(len(df_precios.columns))
display(df_precios.head())
print(df_precios.shape)


32


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-27,0.689889,11.077363,5.569457,10.253315,18.232380,2.889291,2.997112,7.648311,7.029698,21.169470,...,4.974968,3.786834,2.356847,0.633228,5.081141,5.878037,4.041083,5.255739,9.107460,0.732158
2006-02-28,0.683571,11.077363,5.523681,10.253315,18.069374,2.889291,2.975575,7.526620,6.898953,20.800888,...,4.916001,3.735913,2.356847,0.633228,5.156895,5.866035,3.949223,5.278191,9.056106,0.732158
2006-03-01,0.701761,11.077363,5.685424,10.459332,18.093525,2.974550,3.090409,7.719296,6.973042,21.063702,...,5.177694,3.984911,2.447744,0.654747,5.228113,5.806032,4.082763,5.401669,9.131621,0.732158
2006-03-02,0.709611,11.077363,5.679319,10.187660,17.737326,2.940659,3.095670,7.727408,6.958514,20.932297,...,5.207456,4.003597,2.430863,0.655761,5.433351,5.806032,4.079360,5.482491,9.225266,0.732158
2006-03-03,0.717079,11.077363,5.676270,10.119744,17.326797,2.966077,3.066848,7.646282,6.928008,21.114988,...,5.213076,4.016212,2.457483,0.650900,5.533706,5.775309,4.074255,5.421875,9.216203,0.732158


(5281, 32)


In [15]:
#### Investigando cual de las acciones se filtraron afuera

not_included = set(tickers).difference(set(df_precios.columns.to_list()))
    # Ojo: "tickers" ya fue procesado de "AMXB.MX" -> "AMXB"
final_tickers = [t for t in tickers if t not in not_included]

print(not_included)
# df_precios.info()

{'USD', 'NU', 'RDOR3', 'EC', 'B3SA3', 'FUNO11', 'CHILE', 'LTM'}


In [16]:
### Transformando precios en rentabilidad

## Estilo rapido
df_rendis = df_precios.pct_change().dropna()

## Estilo despacio
# df_precios_shifted = df_precios.shift(1)      # cambiar datos en dia t hacia dia t-1
# df_rendis_2 = (df_precios - df_precios_shifted)/df_precios_shifted    # formula: rendimiento = (p_t-p_{t-1})/(p_{t-1})
# df_rendis_2 = df_rendis_2.drop(index=df_rendis_2.index[0]) #Primera fila es NaN

## Verificacion que estilo rapido y despacio son iguales
display(df_rendis.head())
# display(df_precios_shifted.head())
# display(df_rendis.head())
# display(df_rendis_2.head())
# df_rendis.iloc[0].astype(float) == df_rendis_2.iloc[0].astype(float)
# print(df_rendis.iloc[5,0])
# print(df_rendis_2.iloc[5,0])
# df_rendis.astype('float32') == df_rendis_2.astype('float32')    #Todo true


## Conclusion: los 2 metoos para tranformar precios en rentabilidad son casi iguales
    # La unica dieferencia es en su decimales mas profundos, pero son iguales de otras maneras

print(df_rendis.iloc[0].values)
print(df_rendis.iloc[1].values)



,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,-0.009159,0.0,-0.008219,0.000000,-0.008940,0.000000,-0.007186,-0.015911,-0.018599,-0.017411,...,-0.011853,-0.013447,0.000000,0.000000,0.014909,-0.002042,-0.022732,0.004272,-0.005639,0.0
2006-03-01,0.026610,0.0,0.029282,0.020093,0.001337,0.029509,0.038592,0.025599,0.010739,0.012635,...,0.053233,0.066650,0.038567,0.033983,0.013810,-0.010229,0.033814,0.023394,0.008339,0.0
2006-03-02,0.011187,0.0,-0.001074,-0.025974,-0.019687,-0.011393,0.001702,0.001051,-0.002083,-0.006238,...,0.005748,0.004689,-0.006897,0.001549,0.039257,0.000000,-0.000834,0.014962,0.010255,0.0
2006-03-03,0.010523,0.0,-0.000537,-0.006666,-0.023145,0.008643,-0.009310,-0.010498,-0.004384,0.008728,...,0.001079,0.003151,0.010951,-0.007414,0.018470,-0.005292,-0.001251,-0.011056,-0.000982,0.0
2006-03-06,-0.013351,0.0,0.021505,-0.035794,-0.008711,-0.018033,-0.042528,-0.027056,-0.028937,-0.008500,...,-0.035334,-0.040828,0.030383,-0.006950,-0.043992,-0.014379,-0.027975,-0.009524,-0.017371,0.0


[-0.00915864  0.         -0.00821918  0.         -0.00894046  0.
 -0.0071857  -0.01591086 -0.01859894 -0.01741101 -0.0032395  -0.00545607
  0.01845669 -0.00165684  0.         -0.00078054 -0.00218712 -0.02964539
  0.0050851   0.         -0.00729152 -0.00518098 -0.01185267 -0.01344703
  0.          0.          0.0149087  -0.00204176 -0.0227316   0.00427197
 -0.00563872  0.        ]
[ 0.02661004  0.          0.02928177  0.02009277  0.00133656  0.02950861
  0.03859223  0.02559928  0.0107391   0.01263473  0.00755643  0.01341076
 -0.00164767  0.01361066  0.          0.02303801  0.04822468  0.05923986
  0.01349047  0.01841834  0.02387563  0.03819453  0.0532328   0.06664982
  0.03856713  0.03398293  0.01381025 -0.01022893  0.03381429  0.02339399
  0.00833866  0.        ]


In [17]:
#### Analizando los precios

### Hay que buscar si las distribuciones son normales
    # posiblemente hacer log transformacion

### Correlacion entre precios de diferentes acciones
## Calculando covariance matriz
corr_precios = df_precios.corr(method='pearson')
    # Hacer cor(columna_i, columna_j) forall i,j columnas
    # Resulta en una matrix con shape = (num_cols, num_cols)
corr_precios = corr_precios.replace(1.0, np.nan)
    # Las diagionales = corr(col_i,col_i) = 1, pero no es relevante
# print(corr_precios)
# print(type(corr_precios.values))
# print(np.min(corr_precios, axis=1))
# print(np.max(corr_precios, axis=1))


#### Creando categorias de correlacion
    # 0 < corr < 0.33 = baja correlacion = categoria 1
    # 0.33 < corr < 0.67 = media correlacion = categoria 2
    # 0.67 < corr < 1.00 = alta correlacion = categoria 3
### Copiando datos para mantener los originales
corr_precios_classificado = corr_precios.copy()
### Intentos no funcionantes:
    # corr_precios_classificado = corr_precios[abs(corr_precios) < 0.33].replace(r'',1)
    # corr_precios_classificado.query('abs(corr_precios_classificado) < 0.33')
### Creando categorias en datos copiados
    #OJO: la inequality se aplica COLUMNA wise
corr_precios_classificado[(abs(corr_precios_classificado) < 0.33) & (abs(corr_precios_classificado) > 0)] = 1
corr_precios_classificado[(abs(corr_precios_classificado) < 0.67) & (abs(corr_precios_classificado) > 0.33)] = 2
corr_precios_classificado[(abs(corr_precios_classificado) < 1.00) & (abs(corr_precios_classificado) > 0.67)] = 3
### Cambiano los diagonales NaNs a 0
corr_precios_classificado = corr_precios_classificado.fillna(0)
corr_precios_classificado = corr_precios_classificado.astype(int)
# print(type(corr_precios_classificado))  #Es un Dataframe
# display(corr_precios_classificado)



### Analizando la cantidad de baja, media, y alta correlaciones de cada accion
    #OJO: la egualiad 'corr_precios_classificados == 1' se aplica COLUMNA wise
        #ende: 'cantidad' tendra FILAS que sean las COLUMNAS de 'corr_precios_classificados'
            # Se puede verificar con el siguiente experimento:      
                # corr_precios_classificado['rand'] = [3]* corr_precios_classificado.shape[0]
                # display(corr_precios_classificado)
            # Tambien con
                # display(corr_precios_classificado[corr_precios_classificado == 1])
                # display(corr_precios_classificado[corr_precios_classificado == 1]).count()
cantidad = pd.DataFrame()
cantidad['Unos'] = corr_precios_classificado[corr_precios_classificado == 1].count()
cantidad['Dos'] = corr_precios_classificado[corr_precios_classificado == 2].count()
cantidad['Tres'] = corr_precios_classificado[corr_precios_classificado == 3].count()

# display(cantidad)
# display(cantidad[cantidad['Tres'] < 5])
# display(cantidad[cantidad['Tres'] > 20])




## REndimientos
# print(df_rendis.shape)
corr_rendis = df_rendis.corr(method='pearson')

In [18]:

# #### Visualizando matriz de correlacion para precios
# ##### Visualizando correlaciones de precios
# ANCHURA = 50
# ALTURA = 25
# #### Matplotlib, con "subplots"
# fig, axs = plt.subplots(1,2,figsize=(ANCHURA, ALTURA))
# PAD = 0.2
#     # Es necesario mantener este valor igual para que las 2 
#             # imgs se vean igual tamano
# ### Creando figura de correlaciones con valores originales
# img0 = axs[0].imshow(corr_precios, cmap='coolwarm')
# axs[0].set_title('Originales', fontsize=ANCHURA, pad = 20)
# axs[0].set_xticklabels(corr_precios.columns,rotation=90, fontsize=ANCHURA)
# axs[0].set_yticklabels(corr_precios.columns,rotation=0, fontsize=ANCHURA)
# cbar0 = fig.colorbar(
#     img0,
#     ax=axs[0],
#     location='left',
#     fraction=0.05,
#     pad=PAD,
# )
# cbar0.set_label('Valores', fontsize=ANCHURA)
# cbar0.ax.tick_params(labelsize=ANCHURA)

# ### Creando figura de correlaciones con valores categoraizados 
# ## Creando valores discretos en vez de continuo
# from matplotlib.colors import BoundaryNorm, ListedColormap
# colores = ['white','red','yellow','blue']
# cmap = ListedColormap(colores)
# bounds = [-0.5,0.5,1.5,2.5,3.5]  #limits/boundaries de las categorias
# norm = BoundaryNorm(bounds, cmap.N)
# ## Creado figura
# img1 = axs[1].imshow(corr_precios_classificado, cmap=cmap, norm=norm)
#     # Creando la figura
# axs[1].set_title('Categorias', fontsize = ANCHURA, pad = 50)
# axs[1].set_xticklabels(corr_precios_classificado.columns, rotation=90, fontsize=ANCHURA)
# axs[1].set_yticklabels(corr_precios_classificado.columns, rotation=0, fontsize=ANCHURA)
# cbar1 = fig.colorbar(
#     img1, 
#     ax=axs[1], 
#     boundaries=bounds,
#     ticks=[0,1,2,3],
#     fraction = 0.05, #Ratio del color bar a la imagen misma
#     pad = PAD,  # Separacion entre la imagen y el colorbar
# )
# cbar1.set_label('Categorias', fontsize=ANCHURA, rotation=90, labelpad=20)
# cbar1.ax.tick_params(labelsize=ANCHURA, )
# ## Ajustando los subplots
# plt.subplots_adjust(
#     left=0.1,   # margen izquierdo
#         # Aumentar -> Achichar horizontalmente
#     right=0.9,  # margen derecho
#         # Aumentar -> Estrechar horizontalmente
#     top=0.9,    # margen superior
#         # Aumentar -> estrechar verticalmente
#     bottom=0.1, # margen inferior
#         # Aumentar -> achicar verticalmente
#     wspace=0.2, # espacio horizontal entre subplots
#     hspace=0.4  # espacio vertical entre subplots
# )
# plt.show()










# #### Matplotlib, sin "subplots"
# # plt.figure(figsize=(25,12))
# # plt.imshow(
# #     corr_precios_classificado, 
# #     cmap='coolwarm',
# # )
# # cb = plt.colorbar(
# #     label='Categorias', 
# #     location='right',
# #     orientation='vertical',
# # )
# # cb.set_label(
# #     "Intensidad", 
# #     fontdict={'fontsize': 14, 'fontweight': 'bold'}
# # )
# # plt.tick_params(labelsize=20)
# # plt.show()










# #### Matplotlib "matshow"
# # fig = plt.figure(figsize=(ANCHURA,ALTURA))
# # plt.title('Original', fontsize=ANCHURA, pad=10)
# # cax = plt.matshow(corr_precios, cmap = 'coolwarm', fignum=fig.number)
# # cb = plt.colorbar(cax)
# # cb.ax.tick_params(labelsize=ANCHURA)
# #### Otra manera
# # ax.plot(corr_precios)
# # ax.plot(corr_precios_classificado)
# # plt.matshow(corr_precios)
# # plt.matshow(corr_precios_classificado)
# # plt.legend(loc='upper left', fontsize=40)
# # plt.show()







# #### Con seaborn
# # fig, ax = plt.subplots(1,2,figsize=(25,25))
# # sns.heatmap(
# #     corr_precios,
# #     annot=False, fmt='.2f', cmap='coolwarm',
# #     cbar=False, square=True,
# #     ax=ax[0]
# # )

# # sns.heatmap(
# #     corr_precios_classificado,
# #     annot=False, fmt='.2f', cmap='coolwarm',
# #     cbar=True, square=True,
# #     ax=ax[1]
# # )

# # plt.tight_layout()


In [19]:
# #### Visualizando rendimientos - 2 subplots
# final_tickers = [t for t in tickers if t not in not_included]
# EMPRESA_NUM = 5
# VENTANA = -1
# SALTO_X = 180

# #### Rindiendo un subset de las ventanas disponibles
# # display(df_rendis[final_tickers[EMPRESA_NUM]][:VENTANA])

# #### Visuaizaciones independientes
# # plt.figure(figsize=(50,25))
# # plt.plot(df_rendis.index, df_rendis[final_tickers[4]], label=final_tickers[4])
# # plt.figure(figsize=(50,25))
# # plt.plot(df_precios.index, df_precios[final_tickers[4]], label=final_tickers[4])
# # plt.legend()
# # plt.show()

# #### Visualizaciones juntas
# fig, axes = plt.subplots(2 ,1, figsize = (50,25), sharex=False, sharey=False)
#     # Tratar de evitar "sharex=True"
# ### Creando el plot 1
# axes[0].plot(
#     df_rendis.index[:VENTANA], 
#     df_rendis[final_tickers[EMPRESA_NUM]][:VENTANA], 
#     color='blue', 
#     label=final_tickers[EMPRESA_NUM],
#     linestyle=':'   #'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Configurando el plot 1
# axes[0].set_title('Rendimientos', fontsize=50)
# # axes[0].set_xlabel('Fechas', fontsize=50)
# axes[0].set_ylabel('Valor', fontsize=50)
# # axes[0].tick_params(axis='both', labelsize=20)
# axes[0].tick_params(axis='x', labelsize=20, rotation=70) 
# axes[0].tick_params(axis='y', labelsize=20) 
# axes[0].xaxis.set_major_locator(MultipleLocator(SALTO_X))
#     # Se utiliza para hacer saltos en la eje X
# # axes[0].axhline(-1, color='green')    # Dibuja una recta en el valor del primer parametro
# # axes[1].axvline(VENTANA/2, color='purple')    # Dibuja una reta vertical en index del primer parametro
# axes[0].legend(fontsize=50)
# # axes[0].clear()   # Borra figura


# ### Creando el plot 2
# axes[1].plot(
#     df_precios.index[:VENTANA], 
#     df_precios[final_tickers[EMPRESA_NUM]][:VENTANA], 
#     color='red',                        # Color del grafo
#     label=final_tickers[EMPRESA_NUM],    # Se utiliza para legend
#     linestyle="--"
#     )

# ### Confugrando el plot 2
# axes[1].set_title('Precios', fontsize=50)
# axes[1].set_xlabel('Fechas', fontsize=50, labelpad=50)
# axes[1].set_ylabel('Valor', fontsize=50)
# axes[1].tick_params(axis='x', labelsize=20, rotation=70) 
# axes[1].tick_params(axis='y', labelsize=20) 
# # axes[1].axhline(-1, color='green') 
# # axes[1].axvline(VENTANA/2, color='purple')
# axes[1].legend(fontsize=50)
# axes[1].grid(True)
# axes[1].xaxis.set_major_locator(MultipleLocator(SALTO_X))



# ### Ajustando subplots confugraciones
# plt.subplots_adjust(
#     left=0.1,   # margen izquierdo
#         # Aumentar -> Achichar horizontalmente
#     right=0.9,  # margen derecho
#         # Aumentar -> Estrechar horizontalmente
#     top=0.9,    # margen superior
#         # Aumentar -> estrechar verticalmente
#     bottom=0.1, # margen inferior
#         # Aumentar -> achicar verticalmente
#     wspace=0.4, # espacio horizontal entre subplots
#     hspace=0.4  # espacio vertical entre subplots
# )

# plt.show()

In [ ]:
# ##### Visualizando multiples lineas en el mismo grafo
# #### Hiperparametros
# EMPRESA_NUM = 10
# VENTANA = 1000
# SALTO_X = 50


# #### Creando base de los plots
# fig, ax = plt.subplots(figsize=(50,25))

# #### Primer plot
# ax.plot(
#     df_precios.index[:VENTANA], 
#     # df_precios[final_tickers[EMPRESA_NUM]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     df_precios['CEMEXCPO'][:VENTANA],
#     color='blue', 
#     # label=final_tickers[EMPRESA_NUM],
#     label='CEMEX',
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# #### Segundo plot
# ax.plot(
#     df_precios.index[:VENTANA], 
#     df_precios['BBD'][:VENTANA], 
#     color='green', 
#     # label=final_tickers[EMPRESA_NUM],
#     label = 'BBD',
#     linestyle='solid'
# )

# #### Tercer plot
# ax.plot(
#     df_precios.index[:VENTANA], 
#     df_precios['GGB'][:VENTANA], 
#     color='RED', 
#     # label=final_tickers[EMPRESA_NUM],
#     label = 'GGB',
#     linestyle='solid'
# )

# ## Personalizar la base de la figura
# ax.set_title('No Correlacionales', fontsize = 60)
# ax.set_ylabel('Valores', fontsize=50, labelpad = 50)
# ax.set_xlabel('Fechas', fontsize = 50, labelpad = 30)
# ax.tick_params(axis='x', labelsize=20, rotation=70) 
# ax.tick_params(axis='y', labelsize=20) 
# ax.xaxis.set_major_locator(MultipleLocator(SALTO_X))
# ax.legend(fontsize=30)
# ax.grid(True)

# plt.show()



In [21]:
# #### Creando multiple subplots con diferentes acciones
# ### Hiperparametros
# EMPRESA_NUM = 5
# VENTANA = -1
# SALTO_X = 360
# ANCH_SUBPLT = 50
# ALT_SUBPLT = 25

# ### Base de figura
# fig, axs = plt.subplots(2,2, figsize=(ANCH_SUBPLT,ALT_SUBPLT), sharex=False, sharey=False)

# ### primer plot - primera linea
# axs[0,0].plot(
#     df_precios.index[:VENTANA], 
#     # df_precios[final_tickers[EMPRESA_NUM]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     df_precios['CEMEXCPO'][:VENTANA],
#     color='blue', 
#     # label=final_tickers[EMPRESA_NUM],
#     label='CEMEX',
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Cprimer plot - segunda linea
# axs[0,0].plot(
#     df_precios.index[:VENTANA], 
#     df_precios[final_tickers[EMPRESA_NUM]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     color='green', 
#     label=final_tickers[EMPRESA_NUM],
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Confgracion del primer plot
# axs[0,0].set_title('UNO',fontsize = 1.2*ANCH_SUBPLT)
# # axs[0,0].set_xlabel('Fechas', fontsize= ANCH_SUBPLT)
# axs[0,0].set_xticklabels([])
# # axs[0,0].set_ylabel('Valores', fontsize= ANCH_SUBPLT)
# # axs[0,0].tick_params(axis='x', labelsize=0.4*ANCH_SUBPLT, rotation=70) 
# axs[0,0].tick_params(axis='y', labelsize=0.4*ANCH_SUBPLT) 
# axs[0,0].xaxis.set_major_locator(MultipleLocator(SALTO_X))
# axs[0,0].legend(fontsize=0.8*ANCH_SUBPLT)
# axs[0,0].grid(True)



# ### segundo plot - primera linea
# axs[0,1].plot(
#     df_precios.index[:VENTANA], 
#     df_precios['GGB'][:VENTANA],
#     color='blue', 
#     label='GGB',
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### segundo plot -segunda linea
# axs[0,1].plot(
#     df_precios.index[:VENTANA], 
#     df_precios[final_tickers[EMPRESA_NUM+1]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     color='green', 
#     label=final_tickers[EMPRESA_NUM+1],
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Configuracion del segundo plot
# axs[0,1].set_title('DOS',fontsize = 1.2*ANCH_SUBPLT)
# # axs[0,1].set_xlabel('Fechas', fontsize= ANCH_SUBPLT)
# # axs[0,1].set_ylabel('Valores', fontsize= ANCH_SUBPLT)
# # axs[0,1].set_xticks([])   #Borra TODO relacionado
# # axs[0,1].set_yticks([])   #Borra TODO relacionado
# axs[0,1].set_xticklabels([])
# # axs[0,1].set_yticklabels([])
# # axs[0,1].tick_params(axis='x', labelsize=0.4*ANCH_SUBPLT, rotation=70) 
# axs[0,1].tick_params(axis='y', labelsize=0.4*ANCH_SUBPLT) 
# axs[0,1].xaxis.set_major_locator(MultipleLocator(SALTO_X))
# axs[0,1].legend(fontsize=0.8*ANCH_SUBPLT)
# axs[0,1].grid(True)



# ### tercer plot, primera linea
# axs[1,0].plot(
#     df_precios.index[:VENTANA], 
#     df_precios['BBD'][:VENTANA],
#     color='blue', 
#     label='BBD',
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Tercer plot, segunda linea
# axs[1,0].plot(
#     df_precios.index[:VENTANA], 
#     df_precios[final_tickers[EMPRESA_NUM+5]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     color='green', 
#     label=final_tickers[EMPRESA_NUM+5],
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Configuracion del tercer plot
# axs[1,0].set_title('TRES',fontsize = 1.2*ANCH_SUBPLT)
# # axs[1,0].set_xlabel('Fechas', fontsize= ANCH_SUBPLT)
# # axs[1,0].set_ylabel('Valores', fontsize= ANCH_SUBPLT)
# axs[1,0].tick_params(axis='x', labelsize=0.4*ANCH_SUBPLT, rotation=70) 
# axs[1,0].tick_params(axis='y', labelsize=0.4*ANCH_SUBPLT) 
# axs[1,0].xaxis.set_major_locator(MultipleLocator(SALTO_X))
# axs[1,0].legend(fontsize=0.8*ANCH_SUBPLT)
# axs[1,0].grid(True)



# ### Cuarto plot - primera linea
# axs[1,1].plot(
#     df_precios.index[:VENTANA], 
#     df_precios['ABEV'][:VENTANA],
#     color='blue', 
#     label='ABEV',
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Cuarto plot - segunda linea
# axs[1,1].plot(
#     df_precios.index[:VENTANA], 
#     df_precios[final_tickers[EMPRESA_NUM+3]][:VENTANA], #CEMEXCPO, GGB, BBD, ABEV
#     color='green', 
#     label=final_tickers[EMPRESA_NUM+3],
#     linestyle='solid'   ##'-', '--', '-.', ':', 'None', ' ', '', 'solid', 'dashed', 'dashdot', 'dotted'
# )

# ### Configuracion del cuarto plot
# axs[1,1].set_title('CUATRO',fontsize = 1.2*ANCH_SUBPLT)
# # axs[1,1].set_xlabel('Fechas', fontsize= ANCH_SUBPLT)
# # axs[1,1].set_ylabel('Valores', fontsize= ANCH_SUBPLT)
# # axs[1,1].set_yticks([])
# # axs[1,1].set_yticklabels([])
# axs[1,1].tick_params(axis='x', labelsize=0.4*ANCH_SUBPLT, rotation=70) 
# axs[1,1].tick_params(axis='y', labelsize=0.4*ANCH_SUBPLT) 
# axs[1,1].xaxis.set_major_locator(MultipleLocator(SALTO_X))
# axs[1,1].legend(fontsize=0.8*ANCH_SUBPLT)
# axs[1,1].grid(True)



# ### Ajustando subplots confugraciones
# ## Usar este
# plt.subplots_adjust(
#     left=0.07,   # margen izquierdo
#         # Aumentar -> Achichar horizontalmente
#     right=0.9,  # margen derecho
#         # Aumentar -> Estrechar horizontalmente
#     top=0.9,    # margen superior
#         # Aumentar -> estrechar verticalmente
#     bottom=0.15, # margen inferior
#         # Aumentar -> achicar verticalmente
#     wspace=0.05, # espacio horizontal entre subplots
#     hspace=0.1  # espacio vertical entre subplots
# )


# #### Centrado nombre de ejes en forma global
# fig.text(0.48, 0.04, 'Fechas', ha='center', fontsize=ANCH_SUBPLT)   #Eje x
# fig.text(0.04, 0.5, 'Valores', va='center', rotation='vertical', fontsize=ANCH_SUBPLT)  #Eje y


# plt.show()



In [22]:
def box_histograma(df_main: pd.DataFrame, col_name: str, thresh: float, ventana : int = None):
    #### Hiperparametros
    ANCHURA = 25
    ALTURA = 25

    #### Setup
    df = df_main[col_name]
    if ventana is not None: df = df[:ventana]
    ### Normalizaicion
#     scaler = StandardScaler()
#     df = scaler.fit_transform(df)

#     ## Normalizar datos
# df_rendis_norm = pd.DataFrame()
# col_nombres = df_rendis.columns
# scaler = StandardScaler()
# df_rendis_norm[col_nombres] = scaler.fit_transform(df_rendis[col_nombres])
    display(df)
    fig, axs = plt.subplots(2, 1, figsize = (ANCHURA, ALTURA))


    ### Box and whisker 
    axs[0].boxplot(df)
    # Q1 = df.quantile(0.25)
    # Q3 = df.quantile(0.75)
    # IQR = Q3 - Q1
    # thresh = 3
    # lim_inf = Q1 - thresh*IQR
    # lim_sup = Q3 + thresh*IQR
    # outliers = df[
    #     (df < lim_inf)
    # ]
    # print(outliers.index)


    ### Histograma de los valores actuales
    min_redis_norm = min(df)
    max_rendis_norm = max(df)
    num_centros = int(1 + (np.log(df.shape[0])/ np.log(2)))
    centros = np.linspace(min_redis_norm, max_rendis_norm, num_centros)
    cuentas, bordes, parches = axs[1].hist(df, bins=centros)
        # Cuentas = frencuencias absolutas
        # Bordes = intervalos
        # Parhes = configuraciones
    # print(cuentas)
    # print(bordes == centros)
    # print(parches)
    df_stats = pd.DataFrame()
    df_stats['Comienzo'] = bordes[:-1]
    df_stats['Termino'] = bordes[1:]
    df_stats['Frecuencias'] = cuentas
    df_stats['Porcentaje'] = 100 * cuentas/df.shape[0]


    ### Ajustando subplots
    plt.subplots_adjust(
        left=0.07,   # margen izquierdo
            # Aumentar -> Achichar horizontalmente
        right=0.9,  # margen derecho
            # Aumentar -> Estrechar horizontalmente
        top=0.9,    # margen superior
            # Aumentar -> estrechar verticalmente
        bottom=0.15, # margen inferior
            # Aumentar -> achicar verticalmente
        wspace=0.05, # espacio horizontal entre subplots
        hspace=0.1  # espacio vertical entre subplots
    )


    plt.show()
    return df_stats

In [23]:
# #### Histograma e distribucion
#     # Matlab equivalente
# # print(df_rendis_norm.columns.to_list()[:5])
# for i, col in enumerate(df_rendis.columns):
#     if i != 4: continue
#     df_stats = box_histograma(df_rendis, col, 1.5, 50)

# # display(df_stats)

In [24]:
# Datos de ejemplo
# data = {
#     "Producto": ["Laptop", "Mouse", "Teclado", "Laptop", "Mouse"],
#     "Categoría": ["Electrónica", "Accesorios", "Accesorios", "Electrónica", "Accesorios"],
#     "Precio": [3500, 50, 120, 3600, 55]
# }

# # Crear DataFrame
# df = pd.DataFrame(data)

# # Convertir columnas a tipo categórico
# df["Producto"] = df["Producto"].astype("category")
# df["Categoría"] = df["Categoría"].astype("category")

# display(df)
# df.info()

In [25]:
#### Comprobando normalidad con K-S preuba, no log transformacion

## Normalizar datos
df_rendis_norm = pd.DataFrame(index=df_rendis.index)
col_nombres = df_rendis.columns
scaler = StandardScaler()
df_rendis_norm[col_nombres] = scaler.fit_transform(df_rendis[col_nombres])


## Crear dataframe para almacenar datos
sharpiros = pd.DataFrame()
sharpiros['Nombres'] = df_rendis.columns
# sharpiros['rand'] = df_rendis.iloc[0].values
# sharpiros['rand2'] = [0] * len(df_rendis.columns.to_list())
# sharpiros = sharpiros.set_index('Nombres',drop=True)
# display(df_rendis.head())
# display(sharpiros)
# sharpiros.info()

# ## Prueba KS
# from scipy.stats import shapiro
# import random
# ventanas_candidatas = [20,25,30,35,40,45,50,60,75,90, 100]
#     # Se empieza con ventana de 20 pq se necesita al menos 20 datos para hacer preuba concreta de normalidad
# num_dias = df_rendis_norm.shape[0]
# print(num_dias)
#     # Candidatos para el numero de ventanas
# for k in ventanas_candidatas:   # Iterar por candidatos potenciales
#     resultados = []
#     for col in df_rendis_norm.columns:  # ITerar por todas las columnas
#         sum = 0
#         for i in range(30):     # Probary 30 differentes rangos 
#             start_rand = random.randint(0, num_dias - k - 1)
#             # print(start_rand)
#             stat, p = shapiro(df_rendis_norm[col][start_rand:start_rand+k])
#             if p > 0.05: sum += 1
#         resultados.append(sum)
#     sharpiros[f'k{k}'] = resultados


# sharpiros = sharpiros.set_index('Nombres',drop=True)

# display(sharpiros)






In [26]:
##### Analizando Resultados de preubas de normalidad
print(sharpiros.sum(axis=0))


#### Analisis de KS prueba 
# Creo que ninguno de los cariables va a salir normal pq se estan pasando todos los datos historicos que sucedieron
    # Esto incluye momentos buenos y malos
# Ademas, no todos los datos se van a fedear a las vez
    # I.e., 1 data instancia tendra una ventana de datos historicos, la cuales se usan para formar la prediccion del modelo
# Ende, seria mejor medir la normalidad de todas las ventanas posibles a traves de todas las varaibles
    # Esto es mucho...
    # Existen: (df.shape[0] - len(ventana))* len(columnas) = (3856 - 56) * 37 = 140600
# Estrategia: law of large numbers/ central limit theorem


#### Conmlusion
    # Con todos los datos enteros, los activos pierden la prueba
        # Esto tiene sentido pq los activos pasan por periodos impredicibles que no se basan con normalidad
    # A la misma vez, se ve un decremento en las ventanas que pasen la normalidad al subir el tamano de las ventanas
    # Por ende, para estar consistente con los estudios, la ventana seleccionada es k = 30
    # PERO primero intentar con retornos logaritmicos


Nombres    ABEVACAMXBAXIA3BAPBBAS3BBDBIMBOABSACCEMEXCPOCE...
dtype: str


In [27]:
### Rendimientos logaritmicos

# Cambiar los dias por un dia
df_precios_shifted = df_precios.shift(1)


### Formando log
## Formula: log(Rf/Ri) = log(Rf) - log(Ri)
df_rendis_log = np.log(df_precios) - np.log(df_precios_shifted)
df_rendis_log = df_rendis_log.drop(index=df_rendis_log.index[0]) #Primera fila es NaN


### Normalizar datos
df_rendis_log_norm = pd.DataFrame(index=df_rendis.index)
col_nombres = df_rendis_log.columns
scaler = StandardScaler()
df_rendis_log_norm[col_nombres] = scaler.fit_transform(df_rendis_log[col_nombres])

# print(np.std(df_rendis_log_norm[col_nombres], axis=0))


display(df_precios.head())
display(df_precios_shifted.head())
display(df_rendis_log.head())
display(df_rendis_log_norm.head())




# ### Prueba KS
# sharpiros_log = pd.DataFrame()
# sharpiros_log['Nombres'] = df_rendis_log_norm.columns
# ventanas_candidatas = [20,25,30,35,40,45,50,60,75,90,100]
#     # Se empieza con ventana de 20 pq se necesita al menos 20 datos para hacer preuba concreta de normalidad
# num_dias = df_rendis_log_norm.shape[0]
# print(num_dias)
#     # Candidatos para el numero de ventanas
# for k in ventanas_candidatas:   # Iterar por candidatos potenciales
#     resultados = []
#     for col in df_rendis_log_norm.columns:  # ITerar por todas las columnas
#         sum = 0
#         for i in range(30):     # Probary 30 differentes rangos 
#             start_rand = random.randint(0, num_dias - k - 1)
#             # print(start_rand)
#             stat, p = shapiro(df_rendis_log_norm[col][start_rand:start_rand+k])
#             if p > 0.05: sum += 1
#         resultados.append(sum)
#     sharpiros_log[f'k{k}'] = resultados

# sharpiros_log = sharpiros_log.set_index('Nombres',drop=True)
# display(sharpiros_log)


# ## Prueba KS, otra
# from scipy.stats import kstest
# resultados = []
# for col in df_rendis_log_norm.columns:
#     stat, p = kstest(df_rendis_log_norm[col], 'norm')
#     resultados.append(p)

# for i, col in enumerate(df_rendis_log_norm.columns):
#     if i != 0: continue
#     df_stats = box_histograma(df_rendis_log_norm, col, 1.5)

# display(df_stats)

# print(sum([bool(x > 0.1) for x in resultados])) # Despues de log transform todavia es 0
#     # Implicando que ninguno de los variables transformados es normal




,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-27,0.689889,11.077363,5.569457,10.253315,18.232380,2.889291,2.997112,7.648311,7.029698,21.169470,...,4.974968,3.786834,2.356847,0.633228,5.081141,5.878037,4.041083,5.255739,9.107460,0.732158
2006-02-28,0.683571,11.077363,5.523681,10.253315,18.069374,2.889291,2.975575,7.526620,6.898953,20.800888,...,4.916001,3.735913,2.356847,0.633228,5.156895,5.866035,3.949223,5.278191,9.056106,0.732158
2006-03-01,0.701761,11.077363,5.685424,10.459332,18.093525,2.974550,3.090409,7.719296,6.973042,21.063702,...,5.177694,3.984911,2.447744,0.654747,5.228113,5.806032,4.082763,5.401669,9.131621,0.732158
2006-03-02,0.709611,11.077363,5.679319,10.187660,17.737326,2.940659,3.095670,7.727408,6.958514,20.932297,...,5.207456,4.003597,2.430863,0.655761,5.433351,5.806032,4.079360,5.482491,9.225266,0.732158
2006-03-03,0.717079,11.077363,5.676270,10.119744,17.326797,2.966077,3.066848,7.646282,6.928008,21.114988,...,5.213076,4.016212,2.457483,0.650900,5.533706,5.775309,4.074255,5.421875,9.216203,0.732158


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-02-28,0.689889,11.077363,5.569457,10.253315,18.232380,2.889291,2.997112,7.648311,7.029698,21.169470,...,4.974968,3.786834,2.356847,0.633228,5.081141,5.878037,4.041083,5.255739,9.107460,0.732158
2006-03-01,0.683571,11.077363,5.523681,10.253315,18.069374,2.889291,2.975575,7.526620,6.898953,20.800888,...,4.916001,3.735913,2.356847,0.633228,5.156895,5.866035,3.949223,5.278191,9.056106,0.732158
2006-03-02,0.701761,11.077363,5.685424,10.459332,18.093525,2.974550,3.090409,7.719296,6.973042,21.063702,...,5.177694,3.984911,2.447744,0.654747,5.228113,5.806032,4.082763,5.401669,9.131621,0.732158
2006-03-03,0.709611,11.077363,5.679319,10.187660,17.737326,2.940659,3.095670,7.727408,6.958514,20.932297,...,5.207456,4.003597,2.430863,0.655761,5.433351,5.806032,4.079360,5.482491,9.225266,0.732158


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,-0.009201,0.0,-0.008253,0.000000,-0.008981,0.000000,-0.007212,-0.016039,-0.018774,-0.017564,...,-0.011923,-0.013538,0.000000,0.000000,0.014799,-0.002044,-0.022994,0.004263,-0.005655,0.0
2006-03-01,0.026262,0.0,0.028861,0.019894,0.001336,0.029082,0.037866,0.025277,0.010682,0.012556,...,0.051864,0.064523,0.037842,0.033418,0.013716,-0.010282,0.033255,0.023125,0.008304,0.0
2006-03-02,0.011125,0.0,-0.001074,-0.026317,-0.019883,-0.011459,0.001701,0.001050,-0.002086,-0.006258,...,0.005732,0.004678,-0.006920,0.001548,0.038506,0.000000,-0.000834,0.014852,0.010203,0.0
2006-03-03,0.010468,0.0,-0.000537,-0.006689,-0.023417,0.008606,-0.009354,-0.010554,-0.004394,0.008690,...,0.001079,0.003146,0.010891,-0.007441,0.018302,-0.005306,-0.001252,-0.011118,-0.000983,0.0
2006-03-06,-0.013441,0.0,0.021277,-0.036450,-0.008749,-0.018198,-0.043459,-0.027428,-0.029364,-0.008537,...,-0.035973,-0.041685,0.029931,-0.006974,-0.044989,-0.014484,-0.028373,-0.009570,-0.017524,0.0


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,PBR,PBR-A,RENT3,SBSP3,SCCO,SQM,VALE,VIV,WALMEX,WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,-0.432145,-0.041655,-0.478792,-0.010711,-0.480289,-0.014869,-0.254059,-0.884932,-0.974008,-0.676663,...,-0.380930,-0.422133,-0.020894,-0.031427,0.538157,-0.091419,-0.790085,0.197373,-0.358820,-0.036908
2006-03-01,1.182484,-0.041655,1.607799,0.676564,0.039323,1.151380,1.327307,1.341007,0.531260,0.482867,...,1.614763,1.962095,1.418803,1.429830,0.496849,-0.387018,1.120479,1.108344,0.477800,-0.036908
2006-03-02,0.493280,-0.041655,-0.075197,-0.919915,-1.029414,-0.474401,0.058595,0.035762,-0.121183,-0.241401,...,0.171438,0.134256,-0.284181,0.036267,1.442477,-0.018079,-0.037394,0.708785,0.591600,-0.036908
2006-03-03,0.463391,-0.041655,-0.044985,-0.241793,-1.207414,0.330268,-0.329218,-0.589429,-0.239137,0.334048,...,0.025857,0.087453,0.393461,-0.356806,0.671780,-0.208466,-0.051602,-0.545483,-0.078820,-0.036908
2006-03-06,-0.625187,-0.041655,1.181412,-1.269986,-0.468619,-0.744640,-1.525635,-1.498560,-1.515178,-0.329125,...,-1.133360,-1.281824,1.117825,-0.336378,-1.742486,-0.537800,-0.972803,-0.470701,-1.070204,-0.036908


In [28]:
##### OJO: no es necesario denormalizar pq no nos intersa el valor original con los modelos
    # Las estrategias trabajan directamente con los precios originales
    # Modelo 1 predice cual de las acciones va a ser mayor del mediano
    # Modelo 2 predice las posiciones de las acciones

In [29]:
# #### Analizando normalidades de log retornos
# analisis_norm = pd.DataFrame()
# analisis_norm['Tamanos'] = sharpiros_log.columns
# analisis_norm['Regular'] = sharpiros.sum(axis=0).values
# analisis_norm['Log'] = sharpiros_log.sum(axis=0).values
# # print(sharpiros.sum(axis=0))
# # print(sharpiros_log.sum(axis=0))

# display(analisis_norm)

# #### Conclusiones
#     # Aunque el regular tiene mas subsets con "normalidad", el retorno log es considerado un estandar
#         #Por ende, utilizar retorno log
    

# #### Tamano de ventana
#     # Fjellstrom uso 30 dias
#     # Este analisis senala que 30 dias tienen mas normalidad que 50
#     # Ende, utilizar ventana de 30 dias


# Calculando Rendimiento Equilibrado de Portafolio

In [30]:
#### Jugando con diferentes componentes de dataframe

### Revisando una file
# print(type(df_rendis.iloc[0].values))   #np.ndarray
#     # escojiendo todos los renimientos de todas las empresas dentro de un dia 
# print(df_rendis.iloc[0].values)

### Iterando sobre todas las filas en un dataframe
# for idx, fila in df_rendis.iterrows():
#     print(type(idx), type(fila), type(fila.values))    #str, Series, ndarray
#     print(fila.values)
#     break
## Igualmente
# for row in df_rendis.itertuples(index=True):
#   print(row.index, row.values)

### "apply" casi igual que "map"
# print(df_rendis.iloc[0]['ABEV'])
# def procesar(row,num):
#     return num + row['ABEV']

# resultados = df_rendis.apply(procesar,axis=1, args=(5,))
# print(resultados)
# print(type(resultados)) #pd.Series

# ### El de abajo se usa para calculaciones de rendimiento portafolios
# display(df_rendis.head())

# ### Los de abajo se usa para entrenar
# display(df_rendis_norm.head())
# display(df_rendis_log.head())
# display(df_rendis_log_norm.head())




# assert(sum(posiciones) == 1)

#### Formulas relevantes


- 1: $R_t = \sum_i^n w_{i,t-1} * r_{i,t}$
- 2 :$R_t = \sum_i^n \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} * r_{i,t}$
- 3: $C*\sum_i^n | \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} - \frac{\sigma_{tgt}}{\sigma_{i,t-2}} * w_{i,t-2} |$
- 4: $R_t = \sum_i^n \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} * r_{i,t} - C*\sum_i^n | \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} - \frac{\sigma_{tgt}}{\sigma_{i,t-2}} * w_{i,t-2} |$



In [31]:
#### Crear una copia de "df_rendis" para no malograr los originales

df_port = df_rendis.copy()

In [ ]:
#### Haciendo rendimiento de portafolio regular
    # Formula 1
### Creando la formula
def form1(fila, posiciones: list):
    # assert(sum(posiciones)==1)
    return np.dot(fila.values, posiciones)

### Comprobando la correcta aplicacion de la formula
    # Prueba: posiciones = [1,1,...,1]
    # Ende: np.dot(fila.values, posiciones) = sum(fila.values)

### Aplicado la formula
posiciones = [1] * df_rendis.shape[1]   # shape[1] = numero de empresas 
resultados1  = df_rendis.apply(form1, axis=1, args=(posiciones,))
df_port['form1'] = resultados1

resultados = resultados1.values  # Para transformarlo: pd.df -> ndarray

### Comprobando la formula
    # hacer posicones todas = 1 => rendimiento portafolio = sum(1*df_renis) =  sum de cada fila
ver = []
for i in range(df_rendis.shape[0]): #ITerar por cada fila
    fila = df_rendis.iloc[i].values
    suma = np.sum(fila)
    ver.append(suma)
ver = np.array(ver) # Transformar: python list -> nd.array

#### Asegurar que ambos tengan el mismo data type
resultados = resultados.astype(np.float32)
ver = resultados.astype(np.float32)

#### Viendo que sean iguales
print((resultados == ver).all())    # Resultado: True

### conlusion
    # Como np.sum(comp) == df_renids.shape[0], entonces la funcion trabaja



### "apply" casi igual que "map"
# print(df_rendis.iloc[0]['ABEV'])
# def procesar(row,num):
#     return num + row['ABEV']

# resultados = df_rendis.apply(procesar,axis=1, args=(5,))
# print(resultados)
# print(type(resultados)) #pd.Series

True


In [ ]:
#### Calculando EWMSTD (Exponentially Weighted Moving Standar Deviation)
TAMANO_VENTANA = 30
### Hacer despues, es un poco complicado
    # ASumir que ahorita se tiene

### OJO: se necesita hacer 1 para cada activo, no 1 para todos 
for col in final_tickers:
    df_port[f'EWMSD.{col}'] = [1] * df_port.shape[0]   # Poniendo valores randoms


### Extrayendo las columnas con "EWMSD.*"
ewmsd = re.compile(r'EWMSD')
ewmsd_cols = [col for col in df_port.columns if ewmsd.search(col)]
print(ewmsd_cols)   # Nombre de cols que continenen
# display(df_port[ewmsd_cols])

### Extrayendo columnas que no tengan nada que ver con los activos
# print(final_tickers)
patron = r'\b(' + "|".join(map(re.escape, final_tickers)) + r')\b'
    # "re.escape" asegura que las palabras no tengan caracteres 
            # especiales, como "\n" o " "
# print(patron)
reg = re.compile(patron)
cols_diff = [col for col in df_port.columns if not reg.search(col)]
print(cols_diff)



# print(f'num_ewmsd = {len(ewmsd_cols)}, num_activos = {len(final_tickers)}, otros = {len(cols_diff)}')
# df_port.info()
# display(df_port.head())


['EWMSD.ABEV', 'EWMSD.AC', 'EWMSD.AMXB', 'EWMSD.AXIA3', 'EWMSD.BAP', 'EWMSD.BBAS3', 'EWMSD.BBD', 'EWMSD.BIMBOA', 'EWMSD.BSAC', 'EWMSD.CEMEXCPO', 'EWMSD.CENCOSUD', 'EWMSD.CIB', 'EWMSD.FALABELLA', 'EWMSD.FEMSAUBD', 'EWMSD.GCARSOA1', 'EWMSD.GFNORTEO', 'EWMSD.GGB', 'EWMSD.GMEXICOB', 'EWMSD.ISA', 'EWMSD.ITSA4', 'EWMSD.ITUB', 'EWMSD.PAC', 'EWMSD.PBR', 'EWMSD.PBR-A', 'EWMSD.RENT3', 'EWMSD.SBSP3', 'EWMSD.SCCO', 'EWMSD.SQM', 'EWMSD.VALE', 'EWMSD.VIV', 'EWMSD.WALMEX', 'EWMSD.WEGE3']
['form1']



- 2 :$R_t = \sum_i^n \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t} * r_{i,t}$

In [ ]:
#### Calculando Formula 2
    # EN este bloque se va a condesar a 1 activo para simplificar las calculaciones
    # En este bloque se va a hacer dos diferentes pruebas
        # Preuba 1: asumiendo todo EWMSD = 1 y tgt = 1
            # ende: (tgt/EWMSD) * dot(posicones, rendis) = dot(posiciones, rendis)
        # Prueba 2: asuminendo EWMSD = range(0, df_port.shape[0]) y tgt=1
            # Es mas facil para probar


### Formula 2
def form2(fila, posiciones: list, tgt: int = 1):
    # print(fila)
    if fila['EWMSD_1'] == np.nan: return np.nan
        # RECORDAR: todos los valores son np.ndarray de una forma o otra
    return np.dot(fila[df_rendis.columns.to_list()].values, posiciones) * (tgt/fila['EWMSD_1'])
        # OJO: esta formula se aplicara al 'df_port' pero no todas sus columnas
                # Son empresas, ya que incluimos el 'EWMSD'


# #### Preuba 1
df_port['EWMSD'] = [1] * df_port.shape[0]   # Para Formula 2, prueba 1
### Creando un EWMSD shifted para poder usar volares antiguos, ya que se utiliza \sigma_{i,t-1}
df_port['EWMSD_1'] = df_port['EWMSD'].shift(1)
# display(df_port.head())


### Aplicando la formula
posiciones = [1] * df_rendis.shape[1]   # Creando posiciones de 1
resultados2 = df_port.apply(form2, axis=1, args=(posiciones,))  # Aplicando la formula
df_port['form2preuba1'] = resultados2
# print(resultados2)
# print(resultados1)    # recordar: resultado 1 = sum(dot(posicones = 1, empresas))
print((resultados2[1:] == resultados1[1:]).all())   #True => todos son iguales
print(resultados2)


#### Preuba 2
# df_port['EWMSD'] = range(df_port.shape[0])    # Para formula 2, prueba 2
# df_port['EWMSD_1'] = df_port['EWMSD'].shift(1)
# posiciones = [1] * df_rendis.shape[1]   # Creando posiciones de 1
# df_port['form2prueba2'] = df_port.apply(form2, axis=1, args=(posiciones,))


# display(df_port.head())
# tmp = df_port['form1']/df_port['EWMSD_1']
# print(tmp)
# print((tmp.astype(np.float32)[1:] == df_port['form2prueba2'].values.astype(np.float32)[1:]).all())
    # Sale true, entonces la segunda prueba pasa



True
Date
2006-02-28         NaN
2006-03-01    0.713368
2006-03-02    0.139047
2006-03-03   -0.011430
2006-03-06   -0.576577
                ...   
2026-05-22   -0.166073
2026-05-25    0.261212
2026-05-26    0.269468
2026-05-27    0.025991
2026-05-28   -0.280691
Length: 5280, dtype: float64


In [ ]:
#### Agregando posiciones a los diferentes activos
    #I.e., vamos a integrar posicones en dataframe para quitarlas de funcion parametro
for col in final_tickers:
    df_port[f'Pos.{col}'] = [1] * df_port.shape[0]   # Poniendo valores randoms

In [ ]:
# for col in df_port.columns: print(col)

In [ ]:
#### Formula 2
    # En este bloque se va a considerar con todos los activos

### Creando t-1 para EWMSD y posiciones
for col in final_tickers:
    df_port[f'EWMSD_1.{col}'] = df_port[f'EWMSD.{col}'].shift(1)
    df_port[f'Pos_1.{col}']= df_port[f'Pos.{col}'].shift(1)

# Formula 2.2
def form2_2(fila, tgt: int = 1):
    suma = 0
    for col in final_tickers:  #OJO: en "apply", fila son dataseries => usar ".index" para travesar
        contribucion = (tgt/fila[f'EWMSD_1.{col}'])* fila[col] * fila[f'Pos_1.{col}']
        suma+= contribucion
    return suma

resultados = df_port.apply(form2_2, axis=1)
print(resultados)
    # Con valores igual a formula 2.1, preuba 1, tiene los mismo resultados



### Lo de abajo no es necesario pero es SUPER UTIL
# patron = re.compile(fr'(?<!\w){col}(?![\w-])')
#     #"\w" = matches alfabeto, numeros, y guion _
#     #"(?<!X)A" = matches solo si X no esta IMEDIATAMENTE a la izq de A
#     #"A(?!X)" = matches solo si A no es seguido por X immediatamente a la derecha
# noms = [nom for nom in fila.index if patron.search(nom)]
# if len(noms) != 3: print(noms)






C:\Users\JP\AppData\Local\Temp\ipykernel_3156\581756480.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_port[f'Pos_1.{col}']= df_port[f'Pos.{col}'].shift(1)
C:\Users\JP\AppData\Local\Temp\ipykernel_3156\581756480.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_port[f'EWMSD_1.{col}'] = df_port[f'EWMSD.{col}'].shift(1)


Date
2006-02-28         NaN
2006-03-01    0.713368
2006-03-02    0.139047
2006-03-03   -0.011430
2006-03-06   -0.576577
                ...   
2026-05-22   -0.166073
2026-05-25    0.261212
2026-05-26    0.269468
2026-05-27    0.025991
2026-05-28   -0.280691
Length: 5280, dtype: float64



- 3: $C*\sum_i^n | \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} - \frac{\sigma_{tgt}}{\sigma_{i,t-2}} * w_{i,t-2} |$

In [ ]:
#### Formula 3
### Creando t-2 para EWMSD

for col in final_tickers:
    df_port[f'EWMSD_2.{col}'] = df_port[f'EWMSD.{col}'].shift(2)
    df_port[f'Pos_2.{col}'] = df_port[f'Pos.{col}'].shift(2)

# for col in df_port.columns: print(col)

def form3(fila, tgt:float = 1.0, costo: float = 1.0):
    suma = 0
    for col in final_tickers:
        contribucion = abs(((tgt/fila[f'EWMSD_1.{col}'])*fila[f'Pos_1.{col}']) - ((tgt/fila[f'EWMSD_2.{col}'])*fila[f'Pos_2.{col}']))
        suma += costo * contribucion
    return suma


resultados = df_port.apply(form3, axis=1)
print(resultados)

    

C:\Users\JP\AppData\Local\Temp\ipykernel_3156\1441866356.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_port[f'EWMSD_2.{col}'] = df_port[f'EWMSD.{col}'].shift(2)
C:\Users\JP\AppData\Local\Temp\ipykernel_3156\1441866356.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_port[f'Pos_2.{col}'] = df_port[f'Pos.{col}'].shift(2)


Date
2006-02-28    NaN
2006-03-01    NaN
2006-03-02    0.0
2006-03-03    0.0
2006-03-06    0.0
             ... 
2026-05-22    0.0
2026-05-25    0.0
2026-05-26    0.0
2026-05-27    0.0
2026-05-28    0.0
Length: 5280, dtype: float64


- 4: $R_t = \sum_i^n \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} * r_{i,t} - C*\sum_i^n | \frac{\sigma_{tgt}}{\sigma_{i,t-1}} * w_{i,t-1} - \frac{\sigma_{tgt}}{\sigma_{i,t-2}} * w_{i,t-2} |$

In [ ]:
#### Formula 4
def form4(fila, tgt: float = 1.0, costo: float = 1.0):
    suma = 0
    for col in final_tickers:
        cont1 = (tgt/fila[f'EWMSD_1.{col}'])* fila[col] * fila[f'Pos_1.{col}']
        cont2 = costo * abs(((tgt/fila[f'EWMSD_1.{col}'])*fila[f'Pos_1.{col}']) - ((tgt/fila[f'EWMSD_2.{col}'])*fila[f'Pos_2.{col}']))
        suma += cont1 - cont2
    return suma

resultados = df_port.apply(form4, axis=1)
print(resultados)

Date
2006-02-28         NaN
2006-03-01         NaN
2006-03-02    0.139047
2006-03-03   -0.011430
2006-03-06   -0.576577
                ...   
2026-05-22   -0.166073
2026-05-25    0.261212
2026-05-26    0.269468
2026-05-27    0.025991
2026-05-28   -0.280691
Length: 5280, dtype: float64


# Creando datos para MATLAB

In [ ]:
print(df_precios.index)
print(df_rendis.index)
print(df_rendis_log_norm.index)

Index(['2006-02-27', '2006-02-28', '2006-03-01', '2006-03-02', '2006-03-03',
       '2006-03-06', '2006-03-07', '2006-03-08', '2006-03-09', '2006-03-10',
       ...
       '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21',
       '2026-05-22', '2026-05-25', '2026-05-26', '2026-05-27', '2026-05-28'],
      dtype='str', name='Date', length=5281)
Index(['2006-02-28', '2006-03-01', '2006-03-02', '2006-03-03', '2006-03-06',
       '2006-03-07', '2006-03-08', '2006-03-09', '2006-03-10', '2006-03-13',
       ...
       '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21',
       '2026-05-22', '2026-05-25', '2026-05-26', '2026-05-27', '2026-05-28'],
      dtype='str', name='Date', length=5280)
Index(['2006-02-28', '2006-03-01', '2006-03-02', '2006-03-03', '2006-03-06',
       '2006-03-07', '2006-03-08', '2006-03-09', '2006-03-10', '2006-03-13',
       ...
       '2026-05-15', '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21',
       '2026-05-22', '2026-0

In [ ]:
#### Creando columna de medianos
df_rendis_log_norm_matlab = df_rendis_log_norm.copy()   # Manteniendo originales
df_rendis_log_norm_matlab['Median'] = df_rendis_log_norm_matlab.median(axis=1)
    #OJO: mediana(log) = log(mediana)
        # Razon: lg es estrictamente creciente => el orden se mantiene
            # Como mediana solo depende en orden de datos, no su valor, entonces formarmos es conclusion

# display(df_rendis_log_norm_matlab.head())
# df_rendis_log_norm_matlab.info()


In [ ]:
##### Creando los labels
for nombre in df_rendis_log_norm_matlab.columns.to_list()[:-1]:
    comparacion = df_rendis_log_norm_matlab[nombre] > df_rendis_log_norm_matlab['Median']
    df_rendis_log_norm_matlab['Label.' + nombre] = comparacion.astype(int)

df_rendis_log_norm_matlab = df_rendis_log_norm_matlab.drop('Median', axis=1)



In [ ]:
#### Comporbando lo de arriba
print(df_rendis_log_norm_matlab.iloc[0:30,:32].to_numpy().shape)
print(df_rendis_log_norm_matlab.iloc[0:30,:32].index.to_numpy())
display(df_rendis_log_norm_matlab.head())
df_rendis_log_norm_matlab.info()

(30, 32)
['2006-02-28' '2006-03-01' '2006-03-02' '2006-03-03' '2006-03-06'
 '2006-03-07' '2006-03-08' '2006-03-09' '2006-03-10' '2006-03-13'
 '2006-03-14' '2006-03-15' '2006-03-16' '2006-03-17' '2006-03-20'
 '2006-03-21' '2006-03-22' '2006-03-23' '2006-03-24' '2006-03-27'
 '2006-03-28' '2006-03-29' '2006-03-30' '2006-03-31' '2006-04-03'
 '2006-04-04' '2006-04-05' '2006-04-06' '2006-04-07' '2006-04-10']


,ABEV,AC,AMXB,AXIA3,BAP,BBAS3,BBD,BIMBOA,BSAC,CEMEXCPO,...,Label.PBR,Label.PBR-A,Label.RENT3,Label.SBSP3,Label.SCCO,Label.SQM,Label.VALE,Label.VIV,Label.WALMEX,Label.WEGE3
Date,,,,,,,,,,,,,,,,,,,,,
2006-02-28,-0.432145,-0.041655,-0.478792,-0.010711,-0.480289,-0.014869,-0.254059,-0.884932,-0.974008,-0.676663,...,0,0,1,1,1,1,0,1,0,1
2006-03-01,1.182484,-0.041655,1.607799,0.676564,0.039323,1.151380,1.327307,1.341007,0.531260,0.482867,...,1,1,1,1,0,0,1,1,0,0
2006-03-02,0.493280,-0.041655,-0.075197,-0.919915,-1.029414,-0.474401,0.058595,0.035762,-0.121183,-0.241401,...,1,1,0,1,1,0,0,1,1,0
2006-03-03,0.463391,-0.041655,-0.044985,-0.241793,-1.207414,0.330268,-0.329218,-0.589429,-0.239137,0.334048,...,1,1,1,0,1,0,1,0,0,1
2006-03-06,-0.625187,-0.041655,1.181412,-1.269986,-0.468619,-0.744640,-1.525635,-1.498560,-1.515178,-0.329125,...,0,0,1,1,0,1,0,1,0,1


<class 'pandas.DataFrame'>
Index: 5280 entries, 2006-02-28 to 2026-05-28
Data columns (total 64 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ABEV             5280 non-null   float64
 1   AC               5280 non-null   float64
 2   AMXB             5280 non-null   float64
 3   AXIA3            5280 non-null   float64
 4   BAP              5280 non-null   float64
 5   BBAS3            5280 non-null   float64
 6   BBD              5280 non-null   float64
 7   BIMBOA           5280 non-null   float64
 8   BSAC             5280 non-null   float64
 9   CEMEXCPO         5280 non-null   float64
 10  CENCOSUD         5280 non-null   float64
 11  CIB              5280 non-null   float64
 12  FALABELLA        5280 non-null   float64
 13  FEMSAUBD         5280 non-null   float64
 14  GCARSOA1         5280 non-null   float64
 15  GFNORTEO         5280 non-null   float64
 16  GGB              5280 non-null   float64
 17  GMEXICOB       

In [ ]:
# display(df_rendis.head())
# print(df_rendis_log_norm_matlab.iloc[:50].index.to_numpy())
# print(df_rendis_log_norm_matlab.iloc[:50].index.values)


# print(df_rendis['ABEV'].to_numpy())
# print(df_rendis['ABEV'].values)


arr = [str(df_rendis_log_norm_matlab.iloc[50,32:].name)] * 123
ndarr = np.array(arr).astype(str)
ndarr = ndarr.astype(str)
print(ndarr.dtype)

<U10


# Separando datos en train y test
Existen 2 maneras de hacer esta separacion con time series:
* Sliding window validation :  1 mes train y 1 mes test
* Expanding window validation : esta ventana se aumenta gradualmente


In [ ]:
### Para matlab, esto se tendra que hacer en esa plataforma, para Oxford DL
def prepare_batches(train_data_prices, train_data_returns, window_size=30):
    """
    Converts returns and prices into rolling window batches for LSTM training.

    :param train_data_prices: Prices DataFrame (T, num_assets).
    :param train_data_returns: Returns DataFrame (T, num_assets).
    :param window_size: Number of past days used for each prediction.
    :return: Tuple (X, y), where X has shape (samples, window, num_features)
             and y has shape (samples, num_assets).
    """
    X_list, y_list = [], []
    num_assets = train_data_prices.shape[1]

    for i in range(window_size, len(train_data_prices)):
        X_window_prices = train_data_prices.iloc[i-window_size:i].values
        X_window_returns = train_data_returns.iloc[i-window_size:i].values

        X_window = np.hstack([X_window_prices, X_window_returns])  # Concatenate features
        y_target = train_data_returns.iloc[i].values  # Next-day returns

        X_list.append(X_window)
        y_list.append(y_target)

    return np.array(X_list), np.array(y_list)

In [ ]:
##### Procesando los train y test para que esten listo a ser procesados - MATLAB
    #ASUMIENDO: que el conjunto para preprocesar tenga timeseries con columnas SOLO de "input" y "output"
        #I.e., ninguna otra columna que no sea necesaria, por ejemplo "Mediana"

def matlab_procesar(df:pd.DataFrame, tamano_ventana:int = 30):
    datos_ingresos, datos_egresos = [], []
    indices_ingresos, indices_egresos = [], []
    num_registros, num_variables = df.shape
    # print(num_registros, num_variables) #5280, 64
    mitad = int(num_variables/2)

    for i in tqdm(range(num_registros - tamano_ventana), desc='matlabProcesar', position=1, leave=False):
    # for i in range(num_registros - tamano_ventana):
        # print(f"\n\n\n{i}")
        df_ingresos = df.iloc[i:i+tamano_ventana, :mitad]
        df_egresos = df.iloc[i+tamano_ventana, mitad:]

        # print(type(df_ingresos), type(df_egresos))  # pd.Dataframe, pd.Series
        # print(df_ingresos.index.to_numpy(), df_egresos.name)    # Manera de sacar index de pd.Dataframe y pd.Series

        datos_ingresos.append(df_ingresos.to_numpy())
        datos_egresos.append(df_egresos.to_numpy())

        # print(df_ingresos.to_numpy().shape)
        # print(df_egresos.to_numpy().shape)

        indices_ingresos.append(df_ingresos.index.to_numpy())
        indices_egresos.append(df_egresos.name)

        # if i==2: break
    
    datos_ingresos, datos_egresos = np.array(datos_ingresos), np.array(datos_egresos)
    indices_ingresos, indices_egresos = np.array(indices_ingresos), np.array(indices_egresos)
    # print(len(datos_ingresos[0][0]), len(datos_egresos[0]), len(indices_ingresos[0]), len(indices_egresos[0]))

    # print(datos_ingresos.shape, datos_egresos.shape, indices_ingresos.shape, indices_egresos.shape)
        # (3, 30, 32) (3, 32) (3, 30) (3,)      # Pesudo: (3, 30, 32) (3, "1", 32) (3, 30) (3, "1")  
        # (procesados, tamano_ventana, num_acciones),(procesados, num_acciones)
            # Tiene sentido, pq cada 30 registros se usa para predecir 1 registro
                # Y cada registro tiene 32 acciones
            # Podemos pensar en (procesados, num_acciones) como (procesados, "1", num_acciones)
                # Asi se visualiza mejor la relacion de 30 pts para 1 pt 
        # Similar: (3, 30) (3, "1")
            # (procesado, tamano_ventana), (procesado,)
            # 30 pts para 1 pt => existen 30 registros, y cada registro tiene su propio indice

    return indices_ingresos, indices_egresos, datos_ingresos, datos_egresos


In [ ]:
# matlab_procesar(df_rendis_log_norm_matlab)

In [ ]:
##### Sliding window validation
def sliding_window_split(df: pd.DataFrame, train_anos=1, test_anos=1, input_window_size = 30):
    """
    Separa los datos usando la tecnica de sliding window

    :param df: Retornos logaritmicos normalizados
    :param train_anos: tamano de ventana para entrenamiento
    :param test anos: tamano de ventana para prueba
    :return: Lista de (train, test) tuples
    """

    dias_por_ano = 252 #Aproximadamente numero de dias para hacer trading
    dias_train = dias_por_ano * train_anos
    dias_test = dias_por_ano * test_anos

    num_registros = len(df)
    acumulacion = []
    prev = 0

    for train_terminar in tqdm(range(dias_train, num_registros, dias_test), desc='slidingSplit', position=0):
    # for train_terminar in range(dias_train, num_registros, dias_test):
        train_empezar = prev
        test_empezar = train_terminar
        test_terminar = test_empezar + dias_test

        if test_terminar > num_registros:
            print(f'test_terminar: {test_terminar}, num_registros: {num_registros}')    #Es una diferencia de 12 dias
            break #Evitar datos incompletos

        train_data = df.iloc[train_empezar:train_terminar]
        test_data = df.iloc[test_empezar-input_window_size:test_terminar]
            # test_empezar-input_window_size: le restamos "input_window_size" para evitar gaps al hacer predicciones con "test_data"
                # Ya que se necesita al menos "input_window_size" cantidad de registros/data para prediccir 1 dato/ registro
        # print(train_data.shape, test_data.shape)    #(252,64), (282,64) ; 252 = dias_por_ano * train_anos
            #OJO: le restamos el "input_window_size" al entrenamiento pq el modelo va a comerce 
                    # esos "input_window_size" valores antes de hacer su PRIMERA prediccion
            #OJO: al no hacer esto con "train_data", significa que "train_data.shape[0] = test_data.shape[0] - input_window_size"
        trainX_idx, trainY_idx, trainX_data, trainY_data = matlab_procesar(train_data, input_window_size)
            #(Z, 30) (Z,) (Z, 30, 32) (Z, 32)       #Z = train_data.shape[0] - input_window_size
        testX_idx, testY_idx, testX_data, testY_data = matlab_procesar(test_data, input_window_size)
        acumulacion.append((trainX_idx, trainY_idx, trainX_data, trainY_data, testX_idx, testY_idx, testX_data, testY_data))

        prev = train_terminar

        # print(type(trainX_idx), type(trainX_data))  # Ambos son ndarrays
        # print('VIENDO',trainX_idx.shape, trainX_data.shape, testX_idx.shape, testX_data.shape)
        # break
    # print(len(acumulacion)) # 19
    return acumulacion


In [ ]:
matlab_sliding_split = sliding_window_split(df_rendis_log_norm_matlab)

slidingSplit:  95%|█████████▌| 19/20 [00:03<00:00,  5.12it/s]

test_terminar: 5292, num_registros: 5280


In [ ]:
### Explorando fechas con matlab_sliding_split
for tup in matlab_sliding_split:
    print(tup[5][0], tup[5][-1])

2007-02-15 2008-02-01
2008-02-04 2009-01-20
2009-01-21 2010-01-07
2010-01-08 2010-12-27
2010-12-28 2011-12-14
2011-12-15 2012-11-30
2012-12-03 2013-11-19
2013-11-20 2014-11-06
2014-11-07 2015-10-26
2015-10-27 2016-10-12
2016-10-13 2017-09-29
2017-10-02 2018-09-18
2018-09-19 2019-09-05
2019-09-06 2020-08-24
2020-08-25 2021-08-11
2021-08-12 2022-07-29
2022-08-01 2023-07-18
2023-07-19 2024-07-04
2024-07-05 2025-06-24


In [ ]:
###### Expanding window tecnica
def expanding_window_split(df_returns:pd.DataFrame, train_years=1, test_years=1, input_window_size = 30):
    """
    Splits data into train-test sets using expanding training window.

    :param df_returns: Scaled returns DataFrame
    :param train_years: Length of training period in years
    :param test_years: Length of testing period in years
    :return: List of (train, test) tuples
    """
    days_per_year = 252  # Approximate number of trading days per year
    train_days = train_years * days_per_year
    test_days = test_years * days_per_year

    num_samples = len(df_returns)
    split_indices = []
    
    for train_end in tqdm(range(train_days, num_samples, test_days), desc='expandingSplit', position=0):
    # for train_end in range(train_days, num_samples, test_days):
        # Range(start, stop, step)
        train_start = 0  # Always train from the beginning
        test_start = train_end
        test_end = test_start + test_days

        if test_end > num_samples:
            break  # Avoid incomplete last test set

        train_data = df_returns.iloc[train_start:train_end]
        test_data = df_returns.iloc[test_start-input_window_size:test_end]
            #OJO: le restamos el "input_window_size" al entrenamiento pq el modelo va a comerce 
                    # esos "input_window_size" valores antes de hacer su PRIMERA prediccion 

        trainX_idx, trainY_idx, trainX_data, trainY_data = matlab_procesar(train_data, input_window_size)
        testX_idx, testY_idx, testX_data, testY_data = matlab_procesar(test_data, input_window_size)
        split_indices.append((trainX_idx, trainY_idx, trainX_data, trainY_data, testX_idx, testY_idx, testX_data, testY_data))

    return split_indices

# Aplicando separacion a datos para MATLAB

In [ ]:
#### Estos dos son list[tuples[trainX_idx, trainY_idx, trainX_data, trainY_data, testX_idx, testY_idx, testX_data, testX_data]]
matlab_sliding_split = sliding_window_split(df_rendis_log_norm_matlab)
    # Rapido
matlab_expanding_split = expanding_window_split(df_rendis_log_norm_matlab)
    # Despacio

slidingSplit:  95%|█████████▌| 19/20 [00:03<00:00,  5.26it/s]


test_terminar: 5292, num_registros: 5280


expandingSplit:  95%|█████████▌| 19/20 [00:19<00:01,  1.00s/it]


In [ ]:
print(type(matlab_expanding_split), len(matlab_expanding_split))                  #list
print(type(matlab_expanding_split[0]), len(matlab_expanding_split[0]))               #tuple
print(type(matlab_expanding_split[0][0]), len(matlab_expanding_split[0][0]))            #ndarray
print(type(matlab_expanding_split[0][0][221]), len(matlab_expanding_split[0][0][221]))          #ndarray
print(type(matlab_expanding_split[0][2][0][0]), len(matlab_expanding_split[0][2][0][0]))       #str o ndarray, dependiendo si es index o data
# Estrutura: split[i-th tuple][j-th of the 8][k-th of the 222/252 splits][l-th value within a split]


# Tipo de datos: list[tuple(trainX_idx, trainY_idx, trainX_data, trainY_data, testX_idx, testY_idx, testX_data, testY_data)]
    # trainX_idx, trainY_idx, trainX_data, trainY_data = (Z, 30) (Z,) (Z, 30, 32) (Z, 32) - todos ndarrays

print(matlab_sliding_split[0][3][0][:5])
    # el 1er tuple, 3ro array (trainX_data), 1er registro, ultimo 4 time steps
        # OJO: cada registro tiene 30 time steps (pq tamano_ventana = 30) tiene 32 pts (pq hay 32 acciones)
        # Existen 222 (Para train) o 252(para test) registros, almenos para 'sliding_split'

### Verificando las fechas de tests
# for tup in matlab_sliding_split:
#     print(tup[5][0], tup[5][-1])


# for tup in matlab_sliding_split:  # Cuando index este en "pd.Index"
#     print(f'empesar:{tup[1].iloc[0].to_string(index=False)}, terminar:{tup[1].iloc[-1].to_string(index=False)}')
    #OJO: aunque el primer test fecha es '2007-02-15', en realidad 


### Transformando "pd.Index" a "pd.Dataframe"
# print(matlab_expanding_split[0][0].to_frame())
# print(matlab_expanding_split[0][0])


<class 'list'> 19
<class 'tuple'> 8
<class 'numpy.ndarray'> 222
<class 'numpy.ndarray'> 30
<class 'numpy.ndarray'> 32
[1. 1. 0. 0. 0.]


In [ ]:
def dataframes_to_mat(dss_list: list[tuple], mat_filename):
    """
    Convierte una lista de DataFrames a un archivo .mat compatible con MATLAB.
    Cada DataFrame se guarda como una estructura separada.
    """

    if not isinstance(dss_list, list):
        raise TypeError("df_list debe ser una lista de pandas.DataFrame.")
    
    dss_dict = {}
    # print(len(dss_list))
    for i, tup in enumerate(dss_list):
        # print(f'tup_num: {i}')
        idxs_datas = {}
        for idx, ndarr in enumerate(tup):
            # print(f'ndarr_num: {idx}, ndarr_shape: {ndarr.shape}')
            # print(type(df))
            # print(isinstance(df, pd.Index))
            # print(isinstance(df, pd.DataFrame))
            # print((not isinstance(df, pd.DataFrame)) and (not isinstance(df, pd.RangeIndex)))
            if (not isinstance(ndarr, np.ndarray)):
                raise TypeError(f"El elemento en la posición {idx} no es un np.ndarray.")


            # print(ndarr.dtype)



            # Convertir DataFrame a diccionario de columnas → arreglos NumPy
            # df_dict = {}
            # for col in df.columns:
            #     col_data = df[col].to_numpy()

            #     # Convertir strings a objeto MATLAB compatible
            #     if col_data.dtype == object:
            #         # Convertir a lista de strings (MATLAB los interpreta como celdas)
            #         col_data = col_data.astype(str)
            #         # col_data = col_data.astype('U')
                
            #     df_dict[col] = col_data

            
            # Guardar cada DataFrame como estructura: df1, df2, df3...
            idxs_datas[f"idxs_data_{idx}"] = ndarr
        dss_dict[f'ds_{i}'] = idxs_datas

    # return dss_dict

    # Guardar archivo .mat
    scipy.io.savemat(mat_filename, dss_dict)
    print(f"Archivo .mat guardado como: {mat_filename}")

In [ ]:
# dss = dataframes_to_mat(matlab_sliding_split,'matlab_sliding_split.mat')

In [ ]:
# print(dss)

In [ ]:
# dataframes_to_mat(matlab_sliding_split,'matlab_sliding_split.mat')


In [ ]:
print(matlab_sliding_split)

[(array([['2006-02-28', '2006-03-01', '2006-03-02', ..., '2006-04-06',
        '2006-04-07', '2006-04-10'],
       ['2006-03-01', '2006-03-02', '2006-03-03', ..., '2006-04-07',
        '2006-04-10', '2006-04-11'],
       ['2006-03-02', '2006-03-03', '2006-03-06', ..., '2006-04-10',
        '2006-04-11', '2006-04-12'],
       ...,
       ['2007-01-01', '2007-01-02', '2007-01-03', ..., '2007-02-07',
        '2007-02-08', '2007-02-09'],
       ['2007-01-02', '2007-01-03', '2007-01-04', ..., '2007-02-08',
        '2007-02-09', '2007-02-12'],
       ['2007-01-03', '2007-01-04', '2007-01-05', ..., '2007-02-09',
        '2007-02-12', '2007-02-13']], shape=(222, 30), dtype=object), array(['2006-04-11', '2006-04-12', '2006-04-13', '2006-04-14',
       '2006-04-17', '2006-04-18', '2006-04-19', '2006-04-20',
       '2006-04-21', '2006-04-24', '2006-04-25', '2006-04-26',
       '2006-04-27', '2006-04-28', '2006-05-01', '2006-05-02',
       '2006-05-03', '2006-05-04', '2006-05-05', '2006-05-08',
  

In [ ]:
def dataframes_to_mat_expanding(dss_list: list[tuple], mat_filename, parte:int):
    # Igual que el otro, pero hay un limite de 256 MB => splt este data en 2
    """
    Convierte una lista de DataFrames a un archivo .mat compatible con MATLAB.
    Cada DataFrame se guarda como una estructura separada.
    """

    if not isinstance(dss_list, list):
        raise TypeError("df_list debe ser una lista de pandas.DataFrame.")
    
    dss_dict = {}
    # print(len(dss_list))
    for i, tup in enumerate(dss_list):
        if parte == 1:
            if i == 13: break
        else:
            if i <13: continue
        # print(f'tup_num: {i}')
        idxs_datas = {}
        for idx, ndarr in enumerate(tup):
            # print(f'ndarr_num: {idx}, ndarr_shape: {ndarr.shape}')
            # print(type(df))
            # print(isinstance(df, pd.Index))
            # print(isinstance(df, pd.DataFrame))
            # print((not isinstance(df, pd.DataFrame)) and (not isinstance(df, pd.RangeIndex)))
            if (not isinstance(ndarr, np.ndarray)):
                raise TypeError(f"El elemento en la posición {idx} no es un np.ndarray.")


            # print(ndarr.dtype)



            # Convertir DataFrame a diccionario de columnas → arreglos NumPy
            # df_dict = {}
            # for col in df.columns:
            #     col_data = df[col].to_numpy()

            #     # Convertir strings a objeto MATLAB compatible
            #     if col_data.dtype == object:
            #         # Convertir a lista de strings (MATLAB los interpreta como celdas)
            #         col_data = col_data.astype(str)
            #         # col_data = col_data.astype('U')
                
            #     df_dict[col] = col_data

            
            # Guardar cada DataFrame como estructura: df1, df2, df3...
            idxs_datas[f"idxs_data_{idx}"] = ndarr
        dss_dict[f'ds_{i}'] = idxs_datas

    # return dss_dict

    # Guardar archivo .mat
    scipy.io.savemat(mat_filename, dss_dict)
    print(f"Archivo .mat guardado como: {mat_filename}")

In [ ]:

# dataframes_to_mat_expanding(matlab_expanding_split,'matlab_expanding_split_2.mat',2)